In [1]:
import sys
sys.path.insert(0,'/home/ujjwalsrao/project/kaggle-whale/python/')
%load_ext autoreload

In [2]:
%autoreload
from processing.classify.data import data_loader, score_loader
from modeling.classify.model import ResNet, Accuracy
from torch.nn import CrossEntropyLoss, MultiMarginLoss
from utility.optimizer import AdamW
from utility.schedular import CosineLR
from modeling.classify.train import train_model
from modeling.classify.score import score_model

## Training + Validation

- ### 224x224 - finetune

In [3]:
train, valid = data_loader('../../data/boxed/train/','../../data/raw/train.csv', 224, 64)

Train Images: 12766 Valid Images: 2931


In [4]:
name = 'freeze_1_size_224'
model = ResNet(freeze=1).float().cuda()
optimizer = AdamW(model.parameters(), lr=1e-2, weight_decay=1)
schedular = CosineLR(optimizer, T_max=100, T_mult=0.98, eta_min=1e-4)
loss_fn = CrossEntropyLoss()
metric_fn = Accuracy()
load_path = None
save_path = '../../model/classify/experiment-1/'
epochs = 10
batch = 64

In [5]:
arguments = []
arguments += [name]
arguments += [model]
arguments += [train]
arguments += [valid]
arguments += [loss_fn]
arguments += [metric_fn]
arguments += [optimizer]
arguments += [schedular]
arguments += [save_path]
arguments += [load_path]
arguments += [epochs]
arguments += [batch]

In [6]:
train_model(*arguments)

100% 12766/12800 [00:44<00:00, 283.82it/s, trn_ac=7.58, trn_ls=8.68, val_ac=3.10, val_ls=8.69]
100% 12766/12800 [00:46<00:00, 275.05it/s, trn_ac=19.69, trn_ls=6.70, val_ac=9.14, val_ls=7.23]
100% 12766/12800 [01:03<00:00, 180.84it/s, trn_ac=33.42, trn_ls=5.43, val_ac=14.31, val_ls=6.66]
100% 12766/12800 [01:21<00:00, 191.20it/s, trn_ac=46.93, trn_ls=4.30, val_ac=25.26, val_ls=5.77]
100% 12766/12800 [01:21<00:00, 176.72it/s, trn_ac=61.46, trn_ls=3.23, val_ac=34.43, val_ls=5.04]
100% 12766/12800 [01:25<00:00, 149.66it/s, trn_ac=74.92, trn_ls=2.36, val_ac=39.22, val_ls=4.67]
100% 12766/12800 [01:26<00:00, 148.32it/s, trn_ac=90.62, trn_ls=1.48, val_ac=48.75, val_ls=4.19]
100% 12766/12800 [01:17<00:00, 187.58it/s, trn_ac=97.37, trn_ls=0.95, val_ac=53.12, val_ls=3.88]
100% 12766/12800 [00:45<00:00, 281.86it/s, trn_ac=99.45, trn_ls=0.61, val_ac=56.21, val_ls=3.69]
100% 12766/12800 [01:22<00:00, 180.08it/s, trn_ac=99.87, trn_ls=0.43, val_ac=58.03, val_ls=3.61]


- ### 224x224 - retrain

In [7]:
train, valid = data_loader('../../data/boxed/train/','../../data/raw/train.csv', 224, 64)

Train Images: 12766 Valid Images: 2931


In [8]:
name = 'freeze_3_size_224'
model = ResNet(freeze=3).float().cuda()
optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=1)
schedular = CosineLR(optimizer, T_max=100, T_mult=0.98, eta_min=1e-4)
loss_fn = CrossEntropyLoss()
metric_fn = Accuracy()
load_path = '../../model/classify/experiment-1/'
save_path = '../../model/classify/experiment-2/'
epochs = 20
batch = 64

In [9]:
arguments = []
arguments += [name]
arguments += [model]
arguments += [train]
arguments += [valid]
arguments += [loss_fn]
arguments += [metric_fn]
arguments += [optimizer]
arguments += [schedular]
arguments += [save_path]
arguments += [load_path]
arguments += [epochs]
arguments += [batch]

In [10]:
train_model(*arguments)

Model Loaded: Loss: 3.6105


100% 12766/12800 [02:00<00:00, 108.35it/s, trn_ac=81.83, trn_ls=2.08, val_ac=41.32, val_ls=4.87]
100% 12766/12800 [02:03<00:00, 109.59it/s, trn_ac=87.73, trn_ls=1.60, val_ac=45.58, val_ls=4.57]
100% 12766/12800 [02:03<00:00, 119.40it/s, trn_ac=95.36, trn_ls=1.07, val_ac=52.30, val_ls=4.13]
100% 12766/12800 [02:03<00:00, 111.19it/s, trn_ac=99.08, trn_ls=0.67, val_ac=55.88, val_ls=3.80]
100% 12766/12800 [02:03<00:00, 201.40it/s, trn_ac=99.87, trn_ls=0.42, val_ac=59.62, val_ls=3.58]
100% 12766/12800 [02:03<00:00, 110.68it/s, trn_ac=99.95, trn_ls=0.30, val_ac=61.75, val_ls=3.49]
100% 12766/12800 [01:14<00:00, 216.95it/s, trn_ac=99.99, trn_ls=0.22, val_ac=62.77, val_ls=3.43]
100% 12766/12800 [01:07<00:00, 217.82it/s, trn_ac=99.99, trn_ls=0.18, val_ac=64.15, val_ls=3.34]
100% 12766/12800 [01:07<00:00, 218.27it/s, trn_ac=100.00, trn_ls=0.16, val_ac=64.27, val_ls=3.32]
100% 12766/12800 [01:07<00:00, 218.03it/s, trn_ac=100.00, trn_ls=0.14, val_ac=65.00, val_ls=3.29]
100% 12766/12800 [01:07<00:0

- ### 448x448 - finetune

In [11]:
train, valid = data_loader('../../data/boxed/train/','../../data/raw/train.csv', 448, 64)

Train Images: 12766 Valid Images: 2931


In [13]:
name = 'freeze_1_size_448'
model = ResNet(freeze=1).float().cuda()
optimizer = AdamW(model.parameters(), lr=(1e-2)/4, weight_decay=1)
schedular = CosineLR(optimizer, T_max=100, T_mult=0.98, eta_min=(1e-4)/4)
loss_fn = CrossEntropyLoss()
metric_fn = Accuracy()
load_path = '../../model/classify/experiment-2/'
save_path = '../../model/classify/experiment-3/'
epochs = 20
batch = 64

In [14]:
arguments = []
arguments += [name]
arguments += [model]
arguments += [train]
arguments += [valid]
arguments += [loss_fn]
arguments += [metric_fn]
arguments += [optimizer]
arguments += [schedular]
arguments += [save_path]
arguments += [load_path]
arguments += [epochs]
arguments += [batch]

In [15]:
train_model(*arguments)

Model Loaded: Loss: 2.9877


100% 12766/12800 [02:22<00:00, 109.70it/s, trn_ac=55.30, trn_ls=3.87, val_ac=34.52, val_ls=5.40]
100% 12766/12800 [02:22<00:00, 108.86it/s, trn_ac=70.02, trn_ls=2.53, val_ac=48.18, val_ls=4.43]
100% 12766/12800 [02:22<00:00, 109.39it/s, trn_ac=86.98, trn_ls=1.37, val_ac=58.04, val_ls=3.82]
100% 12766/12800 [02:21<00:00, 109.63it/s, trn_ac=97.74, trn_ls=0.60, val_ac=65.67, val_ls=3.24]
100% 12766/12800 [02:21<00:00, 109.66it/s, trn_ac=99.88, trn_ls=0.26, val_ac=69.09, val_ls=3.03]
100% 12766/12800 [02:22<00:00, 109.37it/s, trn_ac=99.99, trn_ls=0.16, val_ac=69.88, val_ls=3.00]
100% 12766/12800 [02:21<00:00, 109.32it/s, trn_ac=100.00, trn_ls=0.12, val_ac=70.62, val_ls=2.96]
100% 12766/12800 [02:21<00:00, 109.43it/s, trn_ac=100.00, trn_ls=0.11, val_ac=72.03, val_ls=2.91]
100% 12766/12800 [02:20<00:00, 109.40it/s, trn_ac=100.00, trn_ls=0.10, val_ac=71.72, val_ls=2.89]
100% 12766/12800 [02:22<00:00, 109.29it/s, trn_ac=100.00, trn_ls=0.09, val_ac=72.23, val_ls=2.90]
100% 12766/12800 [02:21<00

- ### 448x448 - upsample

In [16]:
train, valid = data_loader('../../data/boxed/train/','../../data/raw/train.csv', 448, 64, True, True)

Train Images: 12766 Valid Images: 2931


In [17]:
name = 'freeze_1_size_448'
model = ResNet(freeze=1).float().cuda()
optimizer = AdamW(model.parameters(), lr=(5e-3)/4, weight_decay=1)
schedular = CosineLR(optimizer, T_max=100, T_mult=0.98, eta_min=(1e-5)/4)
loss_fn = CrossEntropyLoss()
metric_fn = Accuracy()
load_path = '../../model/classify/experiment-3/'
save_path = '../../model/classify/experiment-4/'
epochs = 10
batch = 64

In [18]:
arguments = []
arguments += [name]
arguments += [model]
arguments += [train]
arguments += [valid]
arguments += [loss_fn]
arguments += [metric_fn]
arguments += [optimizer]
arguments += [schedular]
arguments += [save_path]
arguments += [load_path]
arguments += [epochs]
arguments += [batch]

In [19]:
train_model(*arguments)

Model Loaded: Loss: 2.7965


100% 12766/12800 [02:23<00:00, 109.90it/s, trn_ac=92.59, trn_ls=2.34, val_ac=51.40, val_ls=4.73]
100% 12766/12800 [02:22<00:00, 109.87it/s, trn_ac=93.63, trn_ls=1.28, val_ac=62.55, val_ls=3.95]
100% 12766/12800 [02:21<00:00, 109.79it/s, trn_ac=97.84, trn_ls=0.55, val_ac=68.97, val_ls=3.47]
100% 12766/12800 [02:21<00:00, 109.78it/s, trn_ac=99.41, trn_ls=0.27, val_ac=72.33, val_ls=3.19]
100% 12766/12800 [02:22<00:00, 108.85it/s, trn_ac=99.77, trn_ls=0.16, val_ac=74.22, val_ls=3.04]
100% 12766/12800 [02:21<00:00, 109.84it/s, trn_ac=99.97, trn_ls=0.12, val_ac=75.32, val_ls=2.96]
100% 12766/12800 [02:22<00:00, 109.21it/s, trn_ac=99.97, trn_ls=0.10, val_ac=76.42, val_ls=2.92]
100% 12766/12800 [02:21<00:00, 109.56it/s, trn_ac=100.00, trn_ls=0.09, val_ac=76.73, val_ls=2.89]
100% 12766/12800 [02:21<00:00, 109.71it/s, trn_ac=100.00, trn_ls=0.08, val_ac=76.99, val_ls=2.87]
100% 12766/12800 [02:23<00:00, 109.82it/s, trn_ac=100.00, trn_ls=0.07, val_ac=76.81, val_ls=2.84]


## Training + Scoring

- ### 224x224 - finetune

In [3]:
train, valid = data_loader('../../data/boxed/train/','../../data/raw/train.csv', 224, 64, False, False)

Train Images: 15697 Valid Images: 2931


In [4]:
name = 'freeze_1_size_224'
model = ResNet(freeze=1).float().cuda()
optimizer = AdamW(model.parameters(), lr=1e-2, weight_decay=1)
schedular = CosineLR(optimizer, T_max=100, T_mult=0.98, eta_min=1e-4)
loss_fn = CrossEntropyLoss()
metric_fn = Accuracy()
load_path = None
save_path = '../../model/classify/experiment-5/'
epochs = 10
batch = 64

In [5]:
arguments = []
arguments += [name]
arguments += [model]
arguments += [train]
arguments += [valid]
arguments += [loss_fn]
arguments += [metric_fn]
arguments += [optimizer]
arguments += [schedular]
arguments += [save_path]
arguments += [load_path]
arguments += [epochs]
arguments += [batch]

In [6]:
train_model(*arguments)

100% 15697/15744 [00:53<00:00, 293.90it/s, trn_ac=6.66, trn_ls=8.61, val_ac=6.50, val_ls=7.49]
100% 15697/15744 [00:53<00:00, 292.60it/s, trn_ac=20.10, trn_ls=6.46, val_ac=18.11, val_ls=5.85]
100% 15697/15744 [00:53<00:00, 293.37it/s, trn_ac=33.85, trn_ls=5.20, val_ac=51.19, val_ls=4.07]
100% 15697/15744 [00:53<00:00, 293.12it/s, trn_ac=49.98, trn_ls=3.99, val_ac=74.97, val_ls=2.72]
100% 15697/15744 [00:53<00:00, 293.48it/s, trn_ac=67.51, trn_ls=2.83, val_ac=93.95, val_ls=1.53]
100% 15697/15744 [00:53<00:00, 291.47it/s, trn_ac=84.25, trn_ls=1.88, val_ac=98.46, val_ls=0.88]
100% 15697/15744 [00:55<00:00, 281.93it/s, trn_ac=95.18, trn_ls=1.16, val_ac=99.69, val_ls=0.44]
100% 15697/15744 [00:54<00:00, 289.44it/s, trn_ac=98.89, trn_ls=0.73, val_ac=99.93, val_ls=0.29]
100% 15697/15744 [00:53<00:00, 293.39it/s, trn_ac=99.70, trn_ls=0.51, val_ac=99.97, val_ls=0.21]
100% 15697/15744 [00:53<00:00, 291.21it/s, trn_ac=99.85, trn_ls=0.39, val_ac=100.00, val_ls=0.16]


- ### 224x224 - retrain

In [7]:
train, valid = data_loader('../../data/boxed/train/','../../data/raw/train.csv', 224, 64, False, False)

Train Images: 15697 Valid Images: 2931


In [8]:
name = 'freeze_3_size_224'
model = ResNet(freeze=3).float().cuda()
optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=1)
schedular = CosineLR(optimizer, T_max=100, T_mult=0.98, eta_min=1e-4)
loss_fn = CrossEntropyLoss()
metric_fn = Accuracy()
load_path = '../../model/classify/experiment-5/'
save_path = '../../model/classify/experiment-6/'
epochs = 20
batch = 64

In [9]:
arguments = []
arguments += [name]
arguments += [model]
arguments += [train]
arguments += [valid]
arguments += [loss_fn]
arguments += [metric_fn]
arguments += [optimizer]
arguments += [schedular]
arguments += [save_path]
arguments += [load_path]
arguments += [epochs]
arguments += [batch]

In [10]:
train_model(*arguments)

Model Loaded: Loss: 0.1626


100% 15697/15744 [01:20<00:00, 195.23it/s, trn_ac=80.53, trn_ls=2.24, val_ac=98.13, val_ls=1.47]
100% 15697/15744 [01:20<00:00, 195.92it/s, trn_ac=89.78, trn_ls=1.47, val_ac=99.86, val_ls=0.60]
100% 15697/15744 [01:19<00:00, 196.66it/s, trn_ac=94.89, trn_ls=0.96, val_ac=99.97, val_ls=0.23]
100% 15697/15744 [01:20<00:00, 195.84it/s, trn_ac=98.69, trn_ls=0.60, val_ac=100.00, val_ls=0.14]
100% 15697/15744 [01:19<00:00, 196.72it/s, trn_ac=99.82, trn_ls=0.40, val_ac=100.00, val_ls=0.10]
100% 15697/15744 [01:20<00:00, 195.25it/s, trn_ac=99.98, trn_ls=0.29, val_ac=100.00, val_ls=0.08]
100% 15697/15744 [01:20<00:00, 195.79it/s, trn_ac=100.00, trn_ls=0.23, val_ac=100.00, val_ls=0.07]
100% 15697/15744 [01:19<00:00, 197.05it/s, trn_ac=100.00, trn_ls=0.19, val_ac=100.00, val_ls=0.06]
100% 15697/15744 [01:20<00:00, 195.63it/s, trn_ac=100.00, trn_ls=0.17, val_ac=100.00, val_ls=0.06]
100% 15697/15744 [01:20<00:00, 194.54it/s, trn_ac=100.00, trn_ls=0.15, val_ac=100.00, val_ls=0.06]
100% 15697/15744 [0

- ### 448x448 - finetune

In [11]:
train, valid = data_loader('../../data/boxed/train/','../../data/raw/train.csv', 448, 64, False, False)

Train Images: 15697 Valid Images: 2931


In [12]:
name = 'freeze_1_size_448'
model = ResNet(freeze=1).float().cuda()
optimizer = AdamW(model.parameters(), lr=(1e-2)/4, weight_decay=1)
schedular = CosineLR(optimizer, T_max=100, T_mult=0.98, eta_min=(1e-4)/4)
loss_fn = CrossEntropyLoss()
metric_fn = Accuracy()
load_path = '../../model/classify/experiment-6/'
save_path = '../../model/classify/experiment-7/'
epochs = 20
batch = 64

In [13]:
arguments = []
arguments += [name]
arguments += [model]
arguments += [train]
arguments += [valid]
arguments += [loss_fn]
arguments += [metric_fn]
arguments += [optimizer]
arguments += [schedular]
arguments += [save_path]
arguments += [load_path]
arguments += [epochs]
arguments += [batch]

In [14]:
train_model(*arguments)

Model Loaded: Loss: 0.029


100% 15697/15744 [02:48<00:00, 107.36it/s, trn_ac=53.87, trn_ls=4.03, val_ac=69.25, val_ls=3.09]
100% 15697/15744 [02:48<00:00, 107.25it/s, trn_ac=76.61, trn_ls=2.19, val_ac=98.50, val_ls=0.79]
100% 15697/15744 [02:47<00:00, 107.31it/s, trn_ac=90.54, trn_ls=1.11, val_ac=99.97, val_ls=0.18]
100% 15697/15744 [02:48<00:00, 107.20it/s, trn_ac=98.08, trn_ls=0.50, val_ac=100.00, val_ls=0.09]
100% 15697/15744 [02:47<00:00, 105.98it/s, trn_ac=99.78, trn_ls=0.25, val_ac=100.00, val_ls=0.06]
100% 15697/15744 [02:48<00:00, 107.26it/s, trn_ac=99.99, trn_ls=0.17, val_ac=100.00, val_ls=0.05]
100% 15697/15744 [02:48<00:00, 106.93it/s, trn_ac=100.00, trn_ls=0.14, val_ac=100.00, val_ls=0.05]
100% 15697/15744 [02:47<00:00, 107.34it/s, trn_ac=100.00, trn_ls=0.12, val_ac=100.00, val_ls=0.05]
100% 15697/15744 [02:49<00:00, 107.33it/s, trn_ac=100.00, trn_ls=0.11, val_ac=100.00, val_ls=0.05]
100% 15697/15744 [02:48<00:00, 105.90it/s, trn_ac=100.00, trn_ls=0.10, val_ac=100.00, val_ls=0.04]
100% 15697/15744 [0

- ### 448x448 - upsample

In [15]:
train, valid = data_loader('../../data/boxed/train/','../../data/raw/train.csv', 448, 64, False, True)

Train Images: 15697 Valid Images: 2931


In [16]:
name = 'freeze_1_size_448'
model = ResNet(freeze=1).float().cuda()
optimizer = AdamW(model.parameters(), lr=(5e-3)/4, weight_decay=1)
schedular = CosineLR(optimizer, T_max=100, T_mult=0.98, eta_min=(1e-5)/4)
loss_fn = CrossEntropyLoss()
metric_fn = Accuracy()
load_path = '../../model/classify/experiment-7/'
save_path = '../../model/classify/experiment-8/'
epochs = 15
batch = 64

In [17]:
arguments = []
arguments += [name]
arguments += [model]
arguments += [train]
arguments += [valid]
arguments += [loss_fn]
arguments += [metric_fn]
arguments += [optimizer]
arguments += [schedular]
arguments += [save_path]
arguments += [load_path]
arguments += [epochs]
arguments += [batch]

In [18]:
train_model(*arguments)

Model Loaded: Loss: 0.0298


100% 15697/15744 [02:48<00:00, 106.74it/s, trn_ac=89.52, trn_ls=2.44, val_ac=94.48, val_ls=1.27]
100% 15697/15744 [02:48<00:00, 107.42it/s, trn_ac=93.95, trn_ls=1.16, val_ac=99.20, val_ls=0.39]
100% 15697/15744 [02:48<00:00, 107.27it/s, trn_ac=98.51, trn_ls=0.45, val_ac=99.80, val_ls=0.14]
100% 15697/15744 [02:48<00:00, 106.27it/s, trn_ac=99.68, trn_ls=0.23, val_ac=99.93, val_ls=0.08]
100% 15697/15744 [02:47<00:00, 106.63it/s, trn_ac=99.87, trn_ls=0.15, val_ac=99.97, val_ls=0.06]
100% 15697/15744 [02:48<00:00, 107.26it/s, trn_ac=99.97, trn_ls=0.12, val_ac=99.97, val_ls=0.05]
100% 15697/15744 [02:47<00:00, 107.05it/s, trn_ac=99.98, trn_ls=0.10, val_ac=100.00, val_ls=0.04]
100% 15697/15744 [02:46<00:00, 107.52it/s, trn_ac=99.99, trn_ls=0.09, val_ac=100.00, val_ls=0.04]
100% 15697/15744 [02:48<00:00, 107.28it/s, trn_ac=100.00, trn_ls=0.08, val_ac=100.00, val_ls=0.04]
100% 15697/15744 [02:48<00:00, 107.41it/s, trn_ac=99.99, trn_ls=0.08, val_ac=100.00, val_ls=0.03]
100% 15697/15744 [02:48<0

- ### scoring

In [19]:
data = score_loader('../../data/boxed/test/', 448, 16, True)

Score Images: 7960


In [20]:
model = ResNet(freeze=False).float().cuda()

In [21]:
score_model(model, data, 16, '../../model/classify/experiment-8/', 5)

Model Loaded: Loss: 0.0303


100% 7960/7968 [00:58<00:00, 136.83it/s]


Records: 7960


100% 7960/7968 [00:58<00:00, 136.31it/s]


Records: 15920


...

Records: 23880


100% 7960/7968 [00:58<00:00, 135.31it/s]


Records: 31840


100% 7960/7968 [00:58<00:00, 135.62it/s]


Records: 39800
